# Cryptocurrency Price Forecasting

## Data Preprocessing

The dataset is prepared for cryptocurrency price forecasting using Prophet and HistGradientBoosting. The preprocessing includes data loading, validation, chronological splitting, feature preparation, and model-specific dataset preparation.

In [1]:
# Import libraries

import pandas as pd
import numpy as np

In [2]:
# Load cleaned dataset

df = pd.read_csv("../data/engineered_data_cleaned.csv")

print("Cleaned dataset loaded successfully.")
print("Dataset shape:", df.shape)

Cleaned dataset loaded successfully.
Dataset shape: (3011, 18)


In [3]:
# Convert Date to datetime

df["Date"] = pd.to_datetime(df["Date"])

print("Date data type:", df["Date"].dtype)

Date data type: datetime64[us]


In [4]:
# Sort data chronologically

df = df.sort_values("Date").reset_index(drop=True)

print("Data sorted chronologically.")
print("First date:", df["Date"].min())
print("Last date:", df["Date"].max())

Data sorted chronologically.
First date: 2013-12-27 00:00:00
Last date: 2022-10-16 00:00:00


In [5]:
# Check data quality

print("Missing values:", df.isnull().sum().sum())
print("Infinite values:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Infinite values: 0
Duplicate rows: 0


In [6]:
# Check date gaps

date_gaps = df["Date"].diff().dt.days

print("Number of date gaps:", (date_gaps > 1).sum())
print("Largest date gap:", date_gaps.max(), "days")

Number of date gaps: 105
Largest date gap: 7.0 days


In [7]:
# Chronological train-test split

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training set:", train_df.shape)
print("Testing set:", test_df.shape)

print("\nTraining period:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nTesting period:")
print(test_df["Date"].min(), "to", test_df["Date"].max())

Training set: (2408, 18)
Testing set: (603, 18)

Training period:
2013-12-27 00:00:00 to 2020-08-04 00:00:00

Testing period:
2020-08-05 00:00:00 to 2022-10-16 00:00:00


In [8]:
# Define features and target

feature_columns = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return",
    "SMA_7",
    "SMA_14",
    "Volatility_7",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Volume_Change"
]

target_column = "Target"

X = df[feature_columns]
y = df[target_column]

print("Number of features:", len(feature_columns))
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Number of features: 16
Feature shape: (3011, 16)
Target shape: (3011,)


In [9]:
# Prepare HistGradientBoosting data

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (2408, 16)
y_train: (2408,)
X_test: (603, 16)
y_test: (603,)


In [10]:
# Prepare Prophet data

prophet_train = train_df[["Date", "Target"]].copy()
prophet_test = test_df[["Date", "Target"]].copy()

prophet_train = prophet_train.rename(
    columns={"Date": "ds", "Target": "y"}
)

prophet_test = prophet_test.rename(
    columns={"Date": "ds", "Target": "y"}
)

print("Prophet training data:")
display(prophet_train.head())

print("\nProphet testing data:")
display(prophet_test.head())

Prophet training data:


,ds,y
0,2013-12-27,727.830017
1,2013-12-28,745.049988
2,2013-12-29,756.130005
3,2013-12-30,754.010010
4,2013-12-31,771.400024



Prophet testing data:


,ds,y
2408,2020-08-05,11779.773257
2409,2020-08-06,11601.472715
2410,2020-08-07,11754.045814
2411,2020-08-08,11675.739343
2412,2020-08-09,11878.111325


In [11]:
# Final preprocessing verification

print("Preprocessing completed successfully.")

print("\nOriginal dataset:", df.shape)
print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)

print("\nHistGradientBoosting:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nProphet:")
print("prophet_train:", prophet_train.shape)
print("prophet_test:", prophet_test.shape)

print("\nMissing values:")
print("HistGradientBoosting X_train:", X_train.isnull().sum().sum())
print("HistGradientBoosting X_test:", X_test.isnull().sum().sum())
print("Prophet train:", prophet_train.isnull().sum().sum())
print("Prophet test:", prophet_test.isnull().sum().sum())

Preprocessing completed successfully.

Original dataset: (3011, 18)
Training dataset: (2408, 18)
Testing dataset: (603, 18)

HistGradientBoosting:
X_train: (2408, 16)
y_train: (2408,)
X_test: (603, 16)
y_test: (603,)

Prophet:
prophet_train: (2408, 2)
prophet_test: (603, 2)

Missing values:
HistGradientBoosting X_train: 0
HistGradientBoosting X_test: 0
Prophet train: 0
Prophet test: 0
